# Figure 1 FC supplement | Supplementary Fig. S1a-d

This notebook evaluates the robustness of FC estimates to the correlation metric, sample size and held-out data across MLP, CNN and Transformer models. It does not recompute SC or FC–IS analyses.

## 1. Inputs and checkpoints

This notebook reuses the checkpoints and deterministic validation data configured for Supplementary Figure 1. If they have not been exported, run the following once in the original analyses:

```python
# MLP notebook
torch.save(models[-1], '../Supplementary_fig_code/checkpoints/mlp_final.pt')

# CNN notebook
torch.save(model_states[-1], '../Supplementary_fig_code/checkpoints/cnn_final.pt')

# Transformer analysis
torch.save(model_states[-1], '../Supplementary_fig_code/checkpoints/transformer_final.pt')
```

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

candidates = [
    Path.cwd(),
    Path.cwd() / 'FC-IS_code' / 'Supplementary_fig_code',
]
CODE_DIR = next((path.resolve() for path in candidates if (path / 'utils' / 'fig1_fc.py').exists()), None)
if CODE_DIR is None:
    raise FileNotFoundError('Could not locate Supplementary_fig_code/utils. Start Jupyter in the project root or notebook folder.')
PROJECT_ROOT = CODE_DIR.parents[1]
sys.path.insert(0, str(CODE_DIR))

from utils.models import MLP, SimpleCNN, TransformerLM, load_state_dict_file
from utils.fig1_fc import (
    SuppFig2Config,
    load_results,
    plot_fig1_fc_supp,
    run_fig1_fc_supp_analysis,
    save_results,
    set_seed,
)

print('Code directory:', CODE_DIR)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2. Configuration

For repeated CNN FC calculations, a fixed seed-selected subset of 256 conv2 feature-map elements is used. This preserves the element-level unit definition while preventing hundreds of repeated 1,600 × 1,600 matrix calculations. The selected indices are saved with the source data.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT_DIR = CODE_DIR / 'checkpoints'
RESULT_DIR = CODE_DIR / 'results' / 'fig1_FC_supp'
OUTPUT_DIR = CODE_DIR / 'outputs' / 'fig1_FC_supp'
for folder in (CHECKPOINT_DIR, RESULT_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

MLP_CHECKPOINT = CHECKPOINT_DIR / 'mlp_final.pt'
CNN_CHECKPOINT = CHECKPOINT_DIR / 'cnn_final.pt'
TRANSFORMER_CHECKPOINT = CHECKPOINT_DIR / 'transformer_final.pt'
RESULT_CACHE = RESULT_DIR / 'fig1_FC_supp_source_data.npz'

MLP_DATA_ROOT = CODE_DIR.parent / 'MLP' / 'data'
CNN_DATA_ROOT = CODE_DIR.parent / 'CNN' / 'data'
TRANSFORMER_DATA_DIR = CODE_DIR.parent / 'Transformer' / 'data' / 'wikitext-2'
DOWNLOAD_IMAGE_DATA = False

config = SuppFig2Config(
    seed=42,
    mlp_n_samples=512,
    cnn_n_samples=256,
    transformer_n_samples=64,
    cnn_element_units=256,
    resampling_repeats=50,
    mlp_sample_sizes=(16, 32, 64, 128, 256),
    cnn_sample_sizes=(8, 16, 32, 64, 128),
    transformer_sample_sizes=(2, 4, 8, 16, 32),
    transformer_layers=(0, 1),
    transformer_chunk_size=2,
    transformer_sc_methods=('signed_mean', 'absolute_mean', 'rms'),
)
set_seed(config.seed)
print(config)

## 3. Deterministic validation loaders

The transformations and word-level vocabulary construction match the current main-analysis code. If the published Transformer uses a stored tokenizer or vocabulary, replace `build_transformer_loader` with that exact artifact before the final run.

In [ ]:
from collections import Counter
from torchvision import datasets, transforms

def build_mlp_loader(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    dataset = datasets.MNIST(
        root=MLP_DATA_ROOT, train=False, transform=transform, download=DOWNLOAD_IMAGE_DATA
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

def build_cnn_loader(batch_size=64):
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    dataset = datasets.MNIST(
        root=CNN_DATA_ROOT, train=False, transform=transform, download=DOWNLOAD_IMAGE_DATA
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

class WordSequenceDataset(Dataset):
    def __init__(self, text, vocab, seq_len=32):
        token_ids = [vocab.get(word, 1) for word in text.split()]
        total = (len(token_ids) // seq_len) * seq_len
        self.data = torch.tensor(token_ids[:total], dtype=torch.long).view(-1, seq_len)

    def __len__(self):
        return max(len(self.data) - 1, 0)

    def __getitem__(self, index):
        return self.data[index], self.data[index + 1]

def build_transformer_loader(batch_size=16, vocab_size=5000, seq_len=32):
    train_path = TRANSFORMER_DATA_DIR / 'wiki.train.tokens'
    validation_path = TRANSFORMER_DATA_DIR / 'wiki.valid.tokens'
    if not train_path.exists() or not validation_path.exists():
        raise FileNotFoundError(
            'WikiText-2 files are missing. Point TRANSFORMER_DATA_DIR to the exact main-analysis data.'
        )
    train_text = train_path.read_text(encoding='utf-8')
    validation_text = validation_path.read_text(encoding='utf-8')
    words = [word for word, _ in Counter(train_text.split()).most_common(vocab_size - 2)]
    vocab = {'<pad>': 0, '<unk>': 1}
    vocab.update({word: index + 2 for index, word in enumerate(words)})
    dataset = WordSequenceDataset(validation_text, vocab, seq_len=seq_len)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0), vocab

## 4. Load models and validation data

In [ ]:
required = [MLP_CHECKPOINT, CNN_CHECKPOINT, TRANSFORMER_CHECKPOINT]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing checkpoint files:\n' + '\n'.join(missing))

mlp_model = MLP(
    input_size=784, hidden_dims=(100, 100), num_classes=10,
    dropout_p=0.0, activation='relu', init_weights=False,
)
cnn_model = SimpleCNN(in_channels=1, activation='relu')
transformer_model = TransformerLM(
    vocab_size=5000, d_model=128, n_heads=4, d_ff=256,
    n_layers=2, seq_len=32, dropout=0.1,
)
load_state_dict_file(mlp_model, str(MLP_CHECKPOINT), map_location='cpu')
load_state_dict_file(cnn_model, str(CNN_CHECKPOINT), map_location='cpu')
load_state_dict_file(transformer_model, str(TRANSFORMER_CHECKPOINT), map_location='cpu')

mlp_loader = build_mlp_loader()
cnn_loader = build_cnn_loader()
transformer_loader, transformer_vocab = build_transformer_loader()
print('Models and data loaders are ready.')

## 5. Run the retained FC robustness analyses

In [ ]:
results = run_fig1_fc_supp_analysis(
    mlp_model=mlp_model,
    mlp_loader=mlp_loader,
    cnn_model=cnn_model,
    cnn_loader=cnn_loader,
    transformer_model=transformer_model,
    transformer_loader=transformer_loader,
    config=config,
    device=DEVICE,
)
save_results(results, RESULT_CACHE)
print('Saved source data:', RESULT_CACHE)

## 6. Plot and export Supplementary Fig. S1a-d

In [ ]:
# results = load_results(RESULT_CACHE)
figure, statistics = plot_fig1_fc_supp(
    results,
    output_prefix=OUTPUT_DIR / 'fig1_FC_supp',
    export_formats=('svg', 'pdf', 'tiff', 'png'),
    dpi=600,
)
display(figure)
statistics

## Reporting checklist

- a: Pearson versus Spearman FC agreement.
- b: FC sampling convergence.
- c: held-out split reproducibility.
- d: layer-wise FC reproducibility.

Report the configured random seed for every stochastic sampling and data-splitting procedure.